In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
local_path = "./models/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    device_map="auto"
)

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 295.27it/s]


In [ ]:
SYSTEM_INSTRUCTION = """
You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract relevant information from messy or incomplete job descriptions.
- Organize content into a clean, standardized JSON structure.
- Improve clarity, grammar, and professionalism while preserving meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Infer missing information conservatively.

Writing Guidelines:
- Use clear, concise, professional language.
- Convert vague phrases into actionable statements.
- Remove redundancy and noise.
- Ensure consistency across sections.
- Use bullet-style phrasing for lists.
"""

In [ ]:
SYSTEM_INSTRUCTION = """
You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract AND rewrite content into a polished, professional format.
- Expand short or vague statements into clear, detailed, and actionable bullet points.
- Improve grammar, clarity, and tone while preserving original meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Do NOT copy sentences directly from input — always rewrite them professionally.

Enhancement Rules:
- Convert short phrases into complete, professional sentences.
- Add clarity by specifying intent (e.g., “handle tickets” → “resolve IT support tickets efficiently within defined SLAs”).
- Use strong action verbs (e.g., manage, ensure, deliver, coordinate, analyze).
- Maintain ATS-friendly language with relevant keywords.
- Avoid vague wording like “do”, “work on”, “handle”.

Writing Style:
- Use concise but complete sentences.
- Each bullet point should be meaningful and self-contained.
- Maintain consistency across all sections.
"""

OUTPUT_SCHEMA = """
{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}
"""

def build_full_prompt(raw_text):
    return f"""
### SYSTEM:
{SYSTEM_INSTRUCTION}

### USER:
Convert the following raw job description into structured JSON.

### INPUT:
{raw_text}

### OUTPUT FORMAT:
Return a fully populated JSON following this schema:

{OUTPUT_SCHEMA}
"""

In [ ]:
content = """Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To provide support for on call escalations and doing root cause analysis of given issue.3. Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.4. To independently resolve tickets within agreed SLA of ticket volume and time.5. To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements: night shifts.
"""


inputs = tokenizer(
    build_full_prompt(content),
    return_tensors="pt",
    truncation=True
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=2000,
    do_sample=False,)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


### SYSTEM:

You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract relevant information from messy or incomplete job descriptions.
- Organize content into a clean, standardized JSON structure.
- Improve clarity, grammar, and professionalism while preserving meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Infer missing information conservatively.

Writing Guidelines:
- Use clear, concise, professional language.
- Convert vague phrases into actionable statements.
- Remove redundancy and noise.
- Ensure consistency across sections.
- Use bullet-style phrasing for lists.


### USER:
Convert the following raw job description into structured JSON.

### INPUT:
Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
I

In [18]:
output = response[response.rfind("{"):response.rfind("}")+1]
print(output)

{
  "job_title": "Administrator (Support & Operations)",
  "location": "Pune, India",
  "industry": "Any Industry",
  "responsibilities": [
    "To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback.",
    "To adhere to quality standards, regulatory requirements and company policies.",
    "To provide support for on call escalations and doing root cause analysis of given issue.",
    "Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.",
    "To independently resolve tickets within agreed SLA of ticket volume and time.",
    "To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases."
  ],
  "requirements": [
    "Must Have Skills: Good to have Skills"
  ],
  "qualifications": [],
  "experience": [],
  "other_requirements": [
    "night shifts"
  ]
}
